In [1]:
import os
import pandas as pd
import numpy as np

print()
print('--- Step 1: Load raw data ---')
df = pd.read_csv('../data/raw/LENDHIST2019_2020_utf8.csv', encoding_errors='ignore')
total_records = df.shape[0]
print(f'Raw records: {total_records}, columns: {df.shape[1]}')

# Normalize string nan to real NaN
for col in ['LEND_DATE', 'RET_DATE']:
    if col in df.columns:
        df[col] = df[col].replace('nan', np.nan)
df = df.replace('nan', np.nan)

print()
print('--- Step 2: Missing-value treatment ---')
user_cols = ['SEX', 'DEPT', 'OCCUPATION']
for col in user_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Unknown')

if 'CODE01' in df.columns:
    df['CODE01'] = pd.to_numeric(df['CODE01'], errors='coerce').fillna(0)

book_cols = ['ABSTRACT', 'SUB', 'CALL_NO', 'AUTHOR', 'PUBLISHER', 'AU']
for col in book_cols:
    if col in df.columns:
        df[col] = df[col].fillna('')

print()
print('--- Step 3: Datetime parsing and features ---')
for col in ['LEND_DATE', 'RET_DATE']:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace(r'(\d{4}-\d{2}-\d{2})(\d{2}:\d{2}:\d{2})', r'\1 \2', regex=True)
        df[col] = pd.to_datetime(df[col], errors='coerce')

df['IS_RETURNED'] = df['RET_DATE'].notnull().astype(int)
df['LEND_DAYS'] = (df['RET_DATE'] - df['LEND_DATE']).dt.total_seconds() / (24 * 3600)
df['LEND_MONTH'] = df['LEND_DATE'].dt.month
df['LEND_HOUR'] = df['LEND_DATE'].dt.hour
df['LEND_DAYOFWEEK'] = df['LEND_DATE'].dt.dayofweek + 1

print()
print('--- Step 4: Data quality checks ---')
invalid_date_count = int(df['LEND_DATE'].isnull().sum())
negative_duration_count = int(((df['IS_RETURNED'] == 1) & (df['LEND_DAYS'] < 0)).sum())
zero_duration_count = int(((df['IS_RETURNED'] == 1) & (df['LEND_DAYS'] == 0)).sum())
dup_count = int(df.duplicated(subset=['USERID', 'BOOK_ID', 'LEND_DATE'], keep='first').sum())

print(f'invalid lend date: {invalid_date_count}')
print(f'returned but duration < 0: {negative_duration_count}')
print(f'returned but duration = 0 (will convert to unreturned): {zero_duration_count}')
print(f'duplicate borrow events: {dup_count}')

print()
print('--- Step 5: Cleaning rules ---')
mask_valid_date = df['LEND_DATE'].notnull()

# Convert zero-duration returned records to unreturned
zero_duration_mask = (df['IS_RETURNED'] == 1) & (df['LEND_DAYS'] == 0)
if zero_duration_mask.any():
    df.loc[zero_duration_mask, 'RET_DATE'] = pd.NaT
    df.loc[zero_duration_mask, 'IS_RETURNED'] = 0
    df.loc[zero_duration_mask, 'LEND_DAYS'] = np.nan

# Keep unreturned rows and rows with positive duration after return
mask_valid_duration = (df['IS_RETURNED'] == 0) | (df['LEND_DAYS'] > 0)

df_cleaned = df[mask_valid_date & mask_valid_duration].copy()
df_cleaned = df_cleaned.drop_duplicates(subset=['USERID', 'BOOK_ID', 'LEND_DATE'], keep='first')

print(f'before: {total_records}')
print(f'after: {df_cleaned.shape[0]}')
print(f'removed: {total_records - df_cleaned.shape[0]}')

print()
print('--- Step 6: Save cleaned data ---')
os.makedirs('../data/processed', exist_ok=True)
output_path = '../data/processed/LENDHIST2019_2020_cleaned.csv'
df_cleaned.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'saved: {output_path}')

display(df_cleaned.head())




--- Step 1: Load raw data ---


Raw records: 99254, columns: 22



--- Step 2: Missing-value treatment ---

--- Step 3: Datetime parsing and features ---



--- Step 4: Data quality checks ---
invalid lend date: 0
returned but duration < 0: 0
returned but duration = 0 (will convert to unreturned): 586
duplicate borrow events: 0

--- Step 5: Cleaning rules ---


before: 99254
after: 99254
removed: 0

--- Step 6: Save cleaned data ---


saved: ../data/processed/LENDHIST2019_2020_cleaned.csv


,USERID,BIRTHYEAR,SEX,DEPT,OCCUPATION,CODE01,REDR_TYPE_NAME,LEND_DATE,RET_DATE,RENEW_TIMES,...,PUBLISHER,ISBN,PUB_YEAR,LOCATION_NAME,DOC_TYPE_NAME,IS_RETURNED,LEND_DAYS,LEND_MONTH,LEND_HOUR,LEND_DAYOFWEEK
0,b1e9675eeacb2d6edcf031bdac1bf0df,1990.0,女,应用金融与行为科学学院,行为经济学,2013,研究生,2020-01-13 21:23:17,2020-06-16 09:03:35,0,...,经济科学出版社,7-5058-1613-6,1998.0,辅助书库一,中文图书,1,154.486319,1,21,1
1,860b7ac7a8a12790da3d77d53e265a21,1998.0,女,公共管理学院,公共管理类(含公共事业管理、行政管理和劳动与社会保障),2018,本科生,2020-12-15 18:10:22,NaT,0,...,武汉大学出版社,7-307-02726-7,1999.0,北区六层,中文图书,0,NaN,12,18,2
2,5a9b0d6f361f4832a93db1ba02a73adc,1996.0,女,会计学院,会计,2018,研究生,2019-11-29 09:46:42,2019-12-30 13:51:18,0,...,西南财经大学出版社,7-81055-433-6,1999.0,北区五层,中文图书,1,31.169861,11,9,5
3,ca2bc2cb7fc2c5e66075aa6b6387c123,1989.0,男,经济学院,产业经济学,2016,研究生,2019-05-17 16:11:00,2019-07-09 16:34:40,0,...,中国人民大学出版社,7-300-02380-0,1997.0,北区四层,中文图书,1,53.016435,5,16,5
4,dca2a9c89a64cbfb95506b69570b9cb6,1996.0,女,法学院,民商法学,2020,研究生,2020-12-28 10:32:48,NaT,0,...,生活.读书.新知三联书店,7-108-00663-4,1994.0,北区二层,中文图书,0,NaN,12,10,1
